# Surgery Log RAG Pipeline: Embeddings, FAISS Vector Store, Retrieval, and LLM Prompt

This notebook starts **after metadata + chunk creation**.

Input expected:

```text
processed_surgery_rag_data/rag_chunks_metadata.csv
```

Pipeline:

```text
rag_chunks_metadata.csv
        ↓
Load chunks + metadata
        ↓
Generate text embeddings
        ↓
Normalize embeddings
        ↓
Store vectors in FAISS
        ↓
Save FAISS index + metadata
        ↓
Load FAISS index
        ↓
Run semantic retrieval
        ↓
Optional metadata filtering
        ↓
Build context for LLM
        ↓
Create final LLM prompt
```

This notebook does **not call an LLM API directly**. It prepares the retrieved context and prompt before LLM generation.

## 1. Install required packages

Run this once in your virtual environment:

```bash
pip install pandas numpy sentence-transformers faiss-cpu
```

If FAISS gives an issue:

```bash
pip install faiss-cpu --no-cache-dir
```

In [16]:
import os
import json
import time
from pathlib import Path
from rank_bm25 import BM25Okapi
import re
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

## 2. Set paths

Recommended structure:

```text
laparoscopic-surgery-rag-assistant/
├── data/
│   └── processed_surgery_rag_data/
│       └── rag_chunks_metadata.csv
├── vector_store/
└── notebooks/
```

In [4]:
PROJECT_DIR = Path.cwd()

CHUNKS_PATH = PROJECT_DIR / "data" / "processed_surgery_rag_data" / "rag_chunks_metadata.csv"

# Fallback path for testing
if not CHUNKS_PATH.exists():
    CHUNKS_PATH = Path("/home/corpadm/my_project/laparoscopic-surgery-rag-assistant/data/processed_surgery_rag_data/rag_chunks_metadata.csv")

VECTOR_STORE_DIR = PROJECT_DIR / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

FAISS_INDEX_PATH = VECTOR_STORE_DIR / "surgery_logs_faiss.index"
METADATA_PATH = VECTOR_STORE_DIR / "surgery_logs_metadata.csv"
EMBEDDINGS_PATH = VECTOR_STORE_DIR / "surgery_logs_embeddings.npy"

print("Chunks path:", CHUNKS_PATH)
print("Vector store folder:", VECTOR_STORE_DIR)

Chunks path: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/data/processed_surgery_rag_data/rag_chunks_metadata.csv
Vector store folder: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/notebooks/vector_store


## 3. Load chunks and metadata

Only the `text` column is embedded.

The remaining columns are metadata and are used after retrieval to identify source case, surgeon, surgery type, source file, and chunk type.

In [5]:
chunks_df = pd.read_csv(CHUNKS_PATH)

required_cols = ["chunk_id", "chunk_type", "case_id", "source_file", "text"]
missing = [c for c in required_cols if c not in chunks_df.columns]

if missing:
    raise ValueError(f"Missing required columns in chunks file: {missing}")

chunks_df["text"] = chunks_df["text"].fillna("").astype(str)
chunks_df = chunks_df[chunks_df["text"].str.strip() != ""].reset_index(drop=True)

print("Total chunks:", len(chunks_df))
display(chunks_df.head())

print("\nChunk type distribution:")
display(chunks_df["chunk_type"].value_counts())

Total chunks: 152


,chunk_id,chunk_type,case_id,source_file,surgery_type,surgeon_name,surgery_date,event_start_time,event_end_time,patient_age,patient_bmi,instrument_names,text
0,Adrenalectomy_2025-10-14_16-12-00__case_summary,case_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,"[""Cobra_Grasper"", ""Fenestrated_Bipolar_Forceps...",Case Adrenalectomy_2025-10-14_16-12-00 is a la...
1,Adrenalectomy_2025-10-14_16-12-00__patient_timing,patient_timing,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,[],Timing and patient details for case Adrenalect...
2,Adrenalectomy_2025-10-14_16-12-00__pedal_activity,pedal_activity,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,[],Pedal activity summary for case Adrenalectomy_...
3,Adrenalectomy_2025-10-14_16-12-00__instrument_...,instrument_summary,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:12:00,2025-10-14 16:55:00,59,20.0,"[""Cobra_Grasper"", ""Fenestrated_Bipolar_Forceps...",Instrument usage summary for case Adrenalectom...
4,Adrenalectomy_2025-10-14_16-12-00__instrument_1,instrument_event,Adrenalectomy_2025-10-14_16-12-00,Adrenalectomy_2025-10-14_16-12-00.json,Adrenalectomy,Dr.Meril M,2025-10-14,2025-10-14 16:55:00,2025-10-14 16:55:00,59,20.0,"[""Monopolar_Cautery_Hook""]",Instrument event for case Adrenalectomy_2025-1...



Chunk type distribution:


chunk_type
instrument_event      88
case_summary          16
patient_timing        16
pedal_activity        16
instrument_summary    16
Name: count, dtype: int64

## 4. Choose embedding model

Good choices:

```text
all-MiniLM-L6-v2     → faster, 384 dimensions
all-mpnet-base-v2    → better quality, 768 dimensions
bge-small-en-v1.5    → strong retrieval model
```

For this project, use:

```text
all-mpnet-base-v2
```

In [6]:
EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Loaded model:", EMBEDDING_MODEL_NAME)

Loaded model: all-mpnet-base-v2


## 5. Generate embeddings

This converts each surgery-log chunk into a dense vector.

In [7]:
texts = chunks_df["text"].tolist()

start = time.time()

embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

embedding_time = time.time() - start
embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)
print(f"Embedding generation time: {embedding_time:.2f} seconds")

Batches: 100%|██████████| 10/10 [00:00<00:00, 10.73it/s]

Embedding shape: (152, 768)
Embedding generation time: 0.94 seconds


## 6. Normalize embeddings

FAISS `IndexFlatIP` uses inner product.

After L2 normalization, inner product behaves like cosine similarity.

In [8]:
faiss.normalize_L2(embeddings)

np.save(EMBEDDINGS_PATH, embeddings)

print("Normalized embeddings saved to:", EMBEDDINGS_PATH)

Normalized embeddings saved to: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/notebooks/vector_store/surgery_logs_embeddings.npy


## 7. Create FAISS vector index

For this project, `IndexFlatIP` is a good choice because it is simple, exact, and reliable.

For very large datasets later, you can explore:

```text
IndexIVFFlat
IndexHNSWFlat
IndexIVFPQ
```

In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created")
print("Embedding dimension:", dimension)
print("Total vectors in index:", index.ntotal)

FAISS index created
Embedding dimension: 768
Total vectors in index: 152


## 8. Save FAISS index and metadata

FAISS stores only vectors.

Metadata is stored separately, so when FAISS returns a vector ID, we can map it back to:

- case ID
- chunk type
- surgery type
- surgeon name
- source JSON file
- original text

In [10]:
faiss.write_index(index, str(FAISS_INDEX_PATH))

metadata_df = chunks_df.copy()
metadata_df["vector_id"] = metadata_df.index

metadata_df.to_csv(METADATA_PATH, index=False)

print("Saved FAISS index:", FAISS_INDEX_PATH)
print("Saved metadata:", METADATA_PATH)

Saved FAISS index: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/notebooks/vector_store/surgery_logs_faiss.index
Saved metadata: /home/corpadm/my_project/laparoscopic-surgery-rag-assistant/notebooks/vector_store/surgery_logs_metadata.csv


## 9. Load vector store again

This simulates production usage.

In [11]:
loaded_index = faiss.read_index(str(FAISS_INDEX_PATH))
loaded_metadata = pd.read_csv(METADATA_PATH)
loaded_embeddings = np.load(EMBEDDINGS_PATH)

print("Loaded vectors:", loaded_index.ntotal)
print("Loaded metadata rows:", len(loaded_metadata))
print("Loaded embeddings shape:", loaded_embeddings.shape)

Loaded vectors: 152
Loaded metadata rows: 152
Loaded embeddings shape: (152, 768)


In [17]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9_]+", " ", text)
    return text.split()


# Prepare BM25 corpus
bm25_corpus = loaded_metadata["text"].fillna("").astype(str).tolist()
tokenized_corpus = [tokenize(doc) for doc in bm25_corpus]

bm25 = BM25Okapi(tokenized_corpus)

## 10. Basic semantic retriever

Steps:

1. Convert query into embedding
2. Normalize query embedding
3. Search FAISS
4. Return top-k chunks with metadata

In [18]:
def retrieve_semantic(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    start = time.time()
    scores, indices = loaded_index.search(query_embedding, top_k)
    retrieval_latency = time.time() - start

    results = loaded_metadata.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    results["rank"] = range(1, len(results) + 1)
    results["retrieval_latency_ms"] = round(retrieval_latency * 1000, 2)

    return results.reset_index(drop=True)


query = "Which instruments were used in Dr MERAI cholecystectomy case?"
results = retrieve_semantic(query, top_k=5)

display(results[[
    "rank",
    "similarity_score",
    "chunk_type",
    "case_id",
    "surgery_type",
    "surgeon_name",
    "source_file",
    "text"
]])

,rank,similarity_score,chunk_type,case_id,surgery_type,surgeon_name,source_file,text
0,1,0.723144,instrument_summary,Cholecystectomy_2025-10-16_16-59-00,Cholecystectomy,Dr.Meril M,Cholecystectomy_2025-10-16_16-59-00.json,Instrument usage summary for case Cholecystect...
1,2,0.714463,instrument_summary,Cholecystectomy_2025-10-14_17-07-00,Cholecystectomy,Dr.Meril M,Cholecystectomy_2025-10-14_17-07-00.json,Instrument usage summary for case Cholecystect...
2,3,0.702301,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument event for case Dr.MERAI_Cholecystec...
3,4,0.696099,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument event for case Dr.MERAI_Cholecystec...
4,5,0.692248,instrument_summary,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument usage summary for case Dr.MERAI_Cho...


## 11. Metadata filtering + vector search

This is closer to production RAG.

Example:

```text
Question: Which instruments were used by Dr.MERAI in Cholecystectomy?
```

Use metadata filters first:

```text
surgery_type = Cholecystectomy
surgeon_name contains MERAI
```

Then search only within those records.

In [19]:
def apply_metadata_filters(df, filters=None):
    if not filters:
        return df.copy()

    filtered = df.copy()

    for col, value in filters.items():
        if value is None or col not in filtered.columns:
            continue

        if isinstance(value, str):
            filtered = filtered[
                filtered[col].fillna("").astype(str).str.contains(value, case=False, na=False)
            ]
        elif isinstance(value, list):
            pattern = "|".join([str(v) for v in value])
            filtered = filtered[
                filtered[col].fillna("").astype(str).str.contains(pattern, case=False, na=False)
            ]
        else:
            filtered = filtered[filtered[col] == value]

    return filtered.reset_index(drop=True)


# def retrieve_with_filters(query, top_k=5, filters=None):
#     filtered_metadata = apply_metadata_filters(loaded_metadata, filters)

#     if filtered_metadata.empty:
#         return pd.DataFrame()

#     vector_ids = filtered_metadata["vector_id"].tolist()
#     filtered_embeddings = loaded_embeddings[vector_ids].astype("float32")

#     temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
#     temp_index.add(filtered_embeddings)

#     query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
#     faiss.normalize_L2(query_embedding)

#     k = min(top_k, len(filtered_metadata))

#     start = time.time()
#     scores, local_indices = temp_index.search(query_embedding, k)
#     retrieval_latency = time.time() - start

#     results = filtered_metadata.iloc[local_indices[0]].copy()
#     results["similarity_score"] = scores[0]
#     results["rank"] = range(1, len(results) + 1)
#     results["retrieval_latency_ms"] = round(retrieval_latency * 1000, 2)

#     return results.reset_index(drop=True)


# query = "Which instruments were used in this surgery?"
# filters = {
#     "surgery_type": "Cholecystectomy",
#     "surgeon_name": "MERAI"
# }

# filtered_results = retrieve_with_filters(query, top_k=5, filters=filters)

# display(filtered_results[[
#     "rank",
#     "similarity_score",
#     "chunk_type",
#     "case_id",
#     "surgery_type",
#     "surgeon_name",
#     "source_file",
#     "text"
# ]])

In [20]:
def normalize_scores(scores):
    scores = np.array(scores, dtype="float32")

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score - min_score == 0:
        return np.ones_like(scores)

    return (scores - min_score) / (max_score - min_score)


def hybrid_search(
    query,
    top_k=5,
    filters=None,
    semantic_weight=0.65,
    keyword_weight=0.35
):
    """
    Full hybrid search:
    1. Apply metadata filters
    2. Run FAISS semantic search
    3. Run BM25 keyword search
    4. Normalize scores
    5. Combine scores
    6. Return final ranked chunks
    """

    # Step 1: metadata filtering
    filtered_metadata = apply_metadata_filters(loaded_metadata, filters)

    if filtered_metadata.empty:
        return pd.DataFrame()

    filtered_vector_ids = filtered_metadata["vector_id"].tolist()

    # Step 2: FAISS semantic search on filtered records
    filtered_embeddings = loaded_embeddings[filtered_vector_ids].astype("float32")

    temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
    temp_index.add(filtered_embeddings)

    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    semantic_k = min(max(top_k * 3, top_k), len(filtered_metadata))

    semantic_scores, semantic_indices = temp_index.search(query_embedding, semantic_k)

    semantic_results = filtered_metadata.iloc[semantic_indices[0]].copy()
    semantic_results["semantic_score"] = semantic_scores[0]

    # Step 3: BM25 keyword search on filtered records
    filtered_texts = filtered_metadata["text"].fillna("").astype(str).tolist()
    filtered_tokens = [tokenize(doc) for doc in filtered_texts]

    temp_bm25 = BM25Okapi(filtered_tokens)
    query_tokens = tokenize(query)

    keyword_scores = temp_bm25.get_scores(query_tokens)

    keyword_results = filtered_metadata.copy()
    keyword_results["keyword_score"] = keyword_scores

    keyword_results = keyword_results.sort_values(
        "keyword_score", ascending=False
    ).head(semantic_k)

    # Step 4: merge semantic + keyword results
    combined = pd.merge(
        semantic_results[
            [
                "vector_id",
                "semantic_score"
            ]
        ],
        keyword_results[
            [
                "vector_id",
                "keyword_score"
            ]
        ],
        on="vector_id",
        how="outer"
    )

    combined["semantic_score"] = combined["semantic_score"].fillna(0)
    combined["keyword_score"] = combined["keyword_score"].fillna(0)

    # Step 5: normalize both scores
    combined["semantic_score_norm"] = normalize_scores(combined["semantic_score"])
    combined["keyword_score_norm"] = normalize_scores(combined["keyword_score"])

    # Step 6: weighted score fusion
    combined["hybrid_score"] = (
        semantic_weight * combined["semantic_score_norm"]
        + keyword_weight * combined["keyword_score_norm"]
    )

    # Step 7: attach metadata
    final_results = pd.merge(
        combined,
        loaded_metadata,
        on="vector_id",
        how="left"
    )

    final_results = final_results.sort_values(
        "hybrid_score", ascending=False
    ).head(top_k)

    final_results["rank"] = range(1, len(final_results) + 1)

    return final_results.reset_index(drop=True)

In [21]:
query = "Which instruments were used in Dr MERAI cholecystectomy case?"

filters = {
    "surgery_type": "Cholecystectomy",
    "surgeon_name": "MERAI"
}

hybrid_results = hybrid_search(
    query=query,
    top_k=5,
    filters=filters,
    semantic_weight=0.65,
    keyword_weight=0.35
)

display(hybrid_results[
    [
        "rank",
        "hybrid_score",
        "semantic_score",
        "keyword_score",
        "chunk_type",
        "case_id",
        "surgery_type",
        "surgeon_name",
        "source_file",
        "text"
    ]
])

,rank,hybrid_score,semantic_score,keyword_score,chunk_type,case_id,surgery_type,surgeon_name,source_file,text
0,1,0.873031,0.680548,4.627231,case_summary,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Case Dr.MERAI_Cholecystectomy_20260521_105445 ...
1,2,0.669964,0.702301,0.529946,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument event for case Dr.MERAI_Cholecystec...
2,3,0.655202,0.696099,0.522619,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument event for case Dr.MERAI_Cholecystec...
3,4,0.640170,0.692248,0.423166,instrument_summary,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument usage summary for case Dr.MERAI_Cho...
4,5,0.563275,0.656060,0.529946,instrument_event,Dr.MERAI_Cholecystectomy_20260521_105445,Cholecystectomy,Dr.MERAI,Dr.MERAI_Cholecystectomy_20260521_105445.json,Instrument event for case Dr.MERAI_Cholecystec...


## 12. Build RAG context

After retrieval, combine top chunks into a clean context block for the LLM.

In [22]:
def build_context(retrieved_df, max_chunks=5):
    if retrieved_df.empty:
        return ""

    context_parts = []

    for _, row in retrieved_df.head(max_chunks).iterrows():
        source = (
            f"Source: case_id={row.get('case_id')}, "
            f"chunk_type={row.get('chunk_type')}, "
            f"surgery_type={row.get('surgery_type')}, "
            f"surgeon={row.get('surgeon_name')}, "
            f"file={row.get('source_file')}"
        )

        text = row.get("text", "")
        context_parts.append(f"{source}\n{text}")

    return "\n\n---\n\n".join(context_parts)


context = build_context(hybrid_results, max_chunks=5)
print(context[:3000])

Source: case_id=Dr.MERAI_Cholecystectomy_20260521_105445, chunk_type=case_summary, surgery_type=Cholecystectomy, surgeon=Dr.MERAI, file=Dr.MERAI_Cholecystectomy_20260521_105445.json
Case Dr.MERAI_Cholecystectomy_20260521_105445 is a laparoscopic surgery log. Surgery type: Cholecystectomy. Surgeon: Dr.MERAI. Patient age: 54, BMI: 62.5. Surgery started: 2026-05-21 09:36:26. Surgery stopped: not recorded. Surgery duration: not recorded. Total events: 293. Clutch pedal presses: 176. Camera pedal presses: 81. Instruments used: Fenestrated_Bipolar_Forceps, Medium_Large_Clip_Applier, Monopolar_Cautery_Hook.

---

Source: case_id=Dr.MERAI_Cholecystectomy_20260521_105445, chunk_type=instrument_event, surgery_type=Cholecystectomy, surgeon=Dr.MERAI, file=Dr.MERAI_Cholecystectomy_20260521_105445.json
Instrument event for case Dr.MERAI_Cholecystectomy_20260521_105445. Instrument name: Monopolar_Cautery_Hook. Side: PrimaryRight. Count: 2. Connected duration: not recorded seconds. Summary time: 2026-

## 13. Create LLM prompt before generation

This is the final step before calling an LLM.

In [23]:
def build_llm_prompt(question, context):
    prompt = f"""
You are a healthcare data assistant for laparoscopic surgery event-log analysis.

Answer the user's question using only the provided retrieved surgery-log context.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say: "Not available in the retrieved surgery logs."
3. Mention case_id and source_file when possible.
4. Keep the answer clear and concise.
5. Do not provide medical advice or clinical decisions.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""
    return prompt.strip()


question = "Which instruments were used in Dr MERAI's cholecystectomy case?"
prompt = build_llm_prompt(question, context)

print(prompt)

You are a healthcare data assistant for laparoscopic surgery event-log analysis.

Answer the user's question using only the provided retrieved surgery-log context.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say: "Not available in the retrieved surgery logs."
3. Mention case_id and source_file when possible.
4. Keep the answer clear and concise.
5. Do not provide medical advice or clinical decisions.

Retrieved Context:
Source: case_id=Dr.MERAI_Cholecystectomy_20260521_105445, chunk_type=case_summary, surgery_type=Cholecystectomy, surgeon=Dr.MERAI, file=Dr.MERAI_Cholecystectomy_20260521_105445.json
Case Dr.MERAI_Cholecystectomy_20260521_105445 is a laparoscopic surgery log. Surgery type: Cholecystectomy. Surgeon: Dr.MERAI. Patient age: 54, BMI: 62.5. Surgery started: 2026-05-21 09:36:26. Surgery stopped: not recorded. Surgery duration: not recorded. Total events: 293. Clutch pedal presses: 176. Camera pedal presses: 81. Instruments used: Fene

## 14. Optional placeholder for LLM call

You can connect the prompt to:

- OpenAI API
- Azure OpenAI
- Gemini
- local LLM through Ollama
- Hugging Face model

For now, this notebook stops before generation.

In [ ]:
def fake_llm_generate(prompt):
    return "LLM generation is not connected in this notebook. The prompt is ready to send to an LLM."

answer = fake_llm_generate(prompt)
print(answer)

## 15. Simple Recall@K evaluation helper

Use this after creating a small test set.

Example test CSV:

```csv
query,expected_case_id
Which instruments were used in Dr MERAI cholecystectomy?,Dr.MERAI_Cholecystectomy_20260521_105445
What was the duration of the 2025 cholecystectomy case?,Cholecystectomy_2025-10-14_17-07-00
```

In [ ]:
def evaluate_recall_at_k(test_df, k=5):
    hits = 0
    rows = []

    for _, item in test_df.iterrows():
        query = item["query"]
        expected_case_id = str(item["expected_case_id"])

        retrieved = retrieve_semantic(query, top_k=k)
        retrieved_case_ids = retrieved["case_id"].astype(str).tolist()

        hit = expected_case_id in retrieved_case_ids
        hits += int(hit)

        rows.append({
            "query": query,
            "expected_case_id": expected_case_id,
            "retrieved_case_ids": retrieved_case_ids,
            f"hit_at_{k}": hit
        })

    recall_at_k = hits / len(test_df) if len(test_df) > 0 else 0
    return recall_at_k, pd.DataFrame(rows)


example_test_df = pd.DataFrame([
    {
        "query": "Which instruments were used in cholecystectomy?",
        "expected_case_id": loaded_metadata.iloc[0]["case_id"]
    }
])

recall, eval_details = evaluate_recall_at_k(example_test_df, k=5)

print(f"Recall@5: {recall:.2f}")
display(eval_details)

# Final output

After running this notebook, you will have:

```text
vector_store/
├── surgery_logs_faiss.index
├── surgery_logs_metadata.csv
└── surgery_logs_embeddings.npy
```

Use these files in your backend/app for RAG retrieval.